# 03 SFT 数据集和 loss：模型到底在学哪几个字

SFT 是 supervised fine-tuning，有标准答案的微调。真正要懂 SFT，必须懂两件事：数据样本怎么被读进来；loss 到底算在哪里。

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, textwrap, re

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts").exists():
    PROJECT_ROOT = Path(r"D:/coding/qwen lorar sft/qwen-local-lora-sft-dpo")

print("项目根目录:", PROJECT_ROOT)
print("学习原则: 本 notebook 默认只读项目文件；所有真实推理/训练开关默认 False。")

def read_text(rel):
    return (PROJECT_ROOT / rel).read_text(encoding="utf-8", errors="replace")

def show_file(rel, start=1, end=None):
    lines = read_text(rel).splitlines()
    if end is None:
        end = len(lines)
    print(f"--- {rel}:{start}-{end} ---")
    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i:03d}: {lines[i-1]}")

def find_lines(rel, keyword, context=4):
    lines = read_text(rel).splitlines()
    hits = []
    for idx, line in enumerate(lines, start=1):
        if keyword in line:
            hits.append(idx)
    if not hits:
        print("没有找到:", keyword)
        return
    for hit in hits:
        start = max(1, hit - context)
        end = min(len(lines), hit + context)
        show_file(rel, start, end)
        print()

def read_jsonl(rel, n=3):
    rows = []
    with (PROJECT_ROOT / rel).open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
            if len(rows) >= n:
                break
    return rows

def count_jsonl(rel):
    with (PROJECT_ROOT / rel).open("r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())

## 1. ChatSFTDataset 是训练数据入口

In [ ]:
find_lines("scripts/train_sft_lora.py", "class ChatSFTDataset", context=8)
find_lines("scripts/train_sft_lora.py", "json.loads", context=8)

这段代码做的事：逐行读取 JSONL，解析成 JSON，找到 `row["messages"]`，调用 `self.encode(...)` 把聊天消息转成 token。

## 2. 最后一条 message 必须是 assistant

In [ ]:
find_lines("scripts/train_sft_lora.py", "Last message must be assistant", context=8)

如果最后不是 assistant，说明这条样本没有标准答案，不能作为 SFT 的监督信号。

## 3. loss 只算 assistant 答案

In [ ]:
find_lines("scripts/train_sft_lora.py", "IGNORE_INDEX", context=10)
find_lines("scripts/train_sft_lora.py", "labels =", context=10)

`IGNORE_INDEX = -100` 是 PyTorch/Transformers 里常见的忽略标记。labels 里为 -100 的位置不会参与 loss。项目把 prompt 部分设成 -100，所以模型不会因为没有“预测题目”而扣分，只会因为 assistant 答案预测不好而扣分。

## 4. DataCollator 为什么需要 padding？

In [ ]:
find_lines("scripts/train_sft_lora.py", "class DataCollatorForChatSFT", context=10)
find_lines("scripts/train_sft_lora.py", "pad_len", context=10)

同一个 batch 里的样本长度可能不同。GPU 喜欢矩阵形状整齐，所以要把短样本补齐：input_ids 用 pad_token_id 补；attention_mask 补 0；labels 补 -100。

## 5. public SFT 和 custom SFT 的差别

In [ ]:
for rel in ["data/processed/sft_train.jsonl", "data/processed/custom_sft_train.jsonl", "data/processed/custom_sft_eval.jsonl"]:
    print(rel, count_jsonl(rel))

public SFT 用来证明工程链路可跑；custom SFT 用来定向修复 LoRA/SFT/DPO 技术概念。面试时不要说 public 数据没用，它的作用是“工程基线”。

## 6. 你要能讲出的底层句子

> SFT 不是把整段聊天都平均学习。代码里先用 chat template 得到 prompt_ids 和 full_ids，再把 prompt 部分 labels 设成 -100，所以 loss 只约束 assistant 标准答案。